In [17]:
import pandas as pd
import numpy as np

# 1. 读取数据
file_path = r"C:\Users\Lenovo\Desktop\上海高考录取数据17-23年\二模成绩.xlsx"
df = pd.read_excel(file_path, engine="openpyxl")



In [18]:
print(df.columns.tolist())  # 检查所有列名
print(df[elec_cols].dtypes)  # 检查分数列的数据类型


['学校', '班级', '姓名', '考号', '总分分数', '语数外分数', '语文分数', '数学分数', '英语分数', '政治分数', '历史分数', '物理分数', '化学分数', '生物(生命科学)分数']
政治分数          float64
历史分数          float64
物理分数          float64
化学分数          float64
生物(生命科学)分数    float64
dtype: object


In [19]:


# 1. 检查所有可能存在的五选科名
possible_elec_cols = ['政治分数', '历史分数', '物理分数', '化学分数', '生物分数', '生物(生命科学)分数']
elec_cols = [col for col in possible_elec_cols if col in df.columns]

# 2. 强制所有五选科分数转 float
for col in elec_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# 3. 再检查
print(df[elec_cols].dtypes)   # 全部应为 float64

# 4. 推断 top2 科目
def top2_subjects(row):
    scores = row[elec_cols].dropna()
    if len(scores) < 2:
        return None
    # 保证全为 float
    scores = scores.astype(float)
    return scores.nlargest(2).index.tolist()

df['top2'] = df.apply(top2_subjects, axis=1)

# 5. 快速映射函数
def quick_map(sub_list, scores_row):
    if not isinstance(sub_list, list):
        return np.nan
    s1, s2 = sub_list
    combo = {s1, s2}

    if combo <= {'物理分数','化学分数'}:
        base = 'IST'
    elif combo <= {'生物分数','化学分数'} or combo <= {'生物(生命科学)分数','化学分数'}:
        base = 'IS'
    elif combo <= {'历史分数','政治分数'}:
        base = 'ENF'
    elif (
        combo & {'物理分数','化学分数','生物分数','生物(生命科学)分数'}
        and combo & {'历史分数','政治分数'}
    ):
        base = 'IN'
    else:
        base = 'XS'

    sel_scores = scores_row[elec_cols].dropna()
    sigma = sel_scores.astype(float).std()
    jp = 'J' if sigma <= 8 else 'P'
    return base + jp

df['MBTI_初判'] = df.apply(lambda x: quick_map(x['top2'], x), axis=1)

out_path = r"C:\Users\Lenovo\Desktop\上海高考录取数据17-23年\二模_MBTI_预测结果.xlsx"
df.to_excel(out_path, index=False)
print('已生成文件：', out_path)


政治分数          float64
历史分数          float64
物理分数          float64
化学分数          float64
生物(生命科学)分数    float64
dtype: object
已生成文件： C:\Users\Lenovo\Desktop\上海高考录取数据17-23年\二模_MBTI_预测结果.xlsx
